# SBI812 — Lecture 6
## Hidden Markov Models and Protein-Region Prediction

### Learning outcomes

By the end of this lecture, a student should be able to:

1. explain why the order of amino acids matters;
2. distinguish observations from hidden states;
3. calculate initial, emission and transition probabilities from counts;
4. use a short Viterbi table to predict hidden states by hand;
5. trace the most probable state path; and
6. evaluate predicted protein regions using known labels.

This lecture uses two observation categories—hydrophobic (`H`) and non-hydrophobic (`P`)—and two hidden states: membrane (`M`) and non-membrane (`N`). All calculations can be completed using a calculator.

## 1. Why an HMM is needed

Earlier classification models treated every row as a separate observation. A protein is different because it is an ordered sequence:

```text
K D A V L I G V L A D K
```

Neighbouring amino acids can form a biological region. A continuous stretch of hydrophobic amino acids may indicate a transmembrane region.

The model must therefore consider both the amino acid at the current position and the likely state of neighbouring positions. An HMM is a machine-learning model for this kind of ordered data.

## 2. What is observed and what is hidden?

Consider the short protein segment:

```text
K  A  V  D
```

The amino acids are observed directly. For this activity, they are grouped into two categories:

| Category | Meaning | Examples |
|---|---|---|
| `H` | Hydrophobic | A, V, I, L, M, F, W, Y, G |
| `P` | Non-hydrophobic | D, E, K, R, S, T, N, Q and others |

Therefore:

```text
Amino acid:   K  A  V  D
Observation:  P  H  H  P
```

The model must predict:

| State | Meaning |
|---|---|
| `N` | Non-membrane |
| `M` | Membrane |

A possible state path is:

```text
Observation:  P  H  H  P
State:        N  M  M  N
```

The observations are visible. The states are hidden until the model predicts them.

## 3. The structure of the model

At each position, the model occupies either state `N` or state `M`.

```mermaid
flowchart LR
    N["N: non-membrane"] -->|"remain N"| N
    N -->|"enter membrane"| M["M: membrane"]
    M -->|"remain M"| M
    M -->|"leave membrane"| N
```

While in a state, the model emits an observation, `H` or `P`.

| Probability type | Question answered |
|---|---|
| Initial | In which state is the sequence likely to begin? |
| Emission | Which observation is likely in the current state? |
| Transition | Which state is likely to occur next? |

## 4. Learning initial probabilities

Suppose the training dataset contains 100 labelled protein segments:

- 80 begin in state `N`;
- 20 begin in state `M`.

$$
P(N\text{ first})=\frac{80}{100}=0.80
$$

$$
P(M\text{ first})=\frac{20}{100}=0.20
$$

| Starting state | Count | Initial probability |
|---|---:|---:|
| `N` | 80 | 0.80 |
| `M` | 20 | 0.20 |

Check: $0.80+0.20=1.00$.

## 5. Learning emission probabilities

### Within non-membrane state N

Suppose positions labelled `N` contain 20 hydrophobic and 80 non-hydrophobic observations:

$$
P(H\mid N)=\frac{20}{100}=0.20
$$

$$
P(P\mid N)=\frac{80}{100}=0.80
$$

### Within membrane state M

Suppose positions labelled `M` contain 90 hydrophobic and 10 non-hydrophobic observations:

$$
P(H\mid M)=\frac{90}{100}=0.90
$$

$$
P(P\mid M)=\frac{10}{100}=0.10
$$

| Current state | Emit `H` | Emit `P` |
|---|---:|---:|
| `N` | 0.20 | 0.80 |
| `M` | 0.90 | 0.10 |

The membrane state strongly favours hydrophobic observations, while the non-membrane state favours non-hydrophobic observations.

## 6. Learning transition probabilities

Suppose 100 transitions begin from state `N`: 70 end in `N` and 30 end in `M`.

$$
P(N_{next}\mid N_{current})=\frac{70}{100}=0.70
$$

$$
P(M_{next}\mid N_{current})=\frac{30}{100}=0.30
$$

Suppose 100 transitions begin from state `M`: 20 end in `N` and 80 end in `M`.

$$
P(N_{next}\mid M_{current})=\frac{20}{100}=0.20
$$

$$
P(M_{next}\mid M_{current})=\frac{80}{100}=0.80
$$

| Current state | Next state `N` | Next state `M` |
|---|---:|---:|
| `N` | 0.70 | 0.30 |
| `M` | 0.20 | 0.80 |

The high probability of remaining in `M` encourages a continuous membrane region rather than isolated membrane predictions.

## 7. The completed trained model

### Initial probabilities

| State | Probability |
|---|---:|
| `N` | 0.80 |
| `M` | 0.20 |

### Emission probabilities

| State | `H` | `P` |
|---|---:|---:|
| `N` | 0.20 | 0.80 |
| `M` | 0.90 | 0.10 |

### Transition probabilities

| From | To `N` | To `M` |
|---|---:|---:|
| `N` | 0.70 | 0.30 |
| `M` | 0.20 | 0.80 |

These three tables form the trained HMM used in the prediction activity.

## 8. Prediction activity

Predict the hidden states for:

```text
Observation:  P  H  H  P
Position:     1  2  3  4
```

At every position, retain the best score for reaching state `N` and the best score for reaching state `M`.

For the first position:

$$
\text{score}=\text{initial probability}\times\text{emission probability}
$$

For later positions:

$$
\text{candidate score}=\text{previous score}\times
\text{transition probability}\times\text{emission probability}
$$

For each current state, compare the two possible routes and retain the larger score.

## 9. Position 1: observation P

### Start in N and emit P

$$
0.80\times0.80=0.6400
$$

### Start in M and emit P

$$
0.20\times0.10=0.0200
$$

| Position 1 state | Score | Previous state |
|---|---:|---|
| `N` | **0.6400** | Start |
| `M` | **0.0200** | Start |

`N` currently has the larger score, but the complete sequence must still be examined.

## 10. Position 2: observation H

### Best route into current state N

Route `N → N`:

$$
0.6400\times0.70\times0.20=0.0896
$$

Route `M → N`:

$$
0.0200\times0.20\times0.20=0.0008
$$

Keep the larger score:

$$
\boxed{N_2=0.0896\text{, from }N_1}
$$

### Best route into current state M

Route `N → M`:

$$
0.6400\times0.30\times0.90=0.1728
$$

Route `M → M`:

$$
0.0200\times0.80\times0.90=0.0144
$$

$$
\boxed{M_2=0.1728\text{, from }N_1}
$$

| Position 2 state | Best score | Came from |
|---|---:|---|
| `N` | 0.0896 | `N` |
| `M` | 0.1728 | `N` |

## 11. Position 3: observation H

### Best route into current state N

$$
N\rightarrow N: 0.0896\times0.70\times0.20=0.012544
$$

$$
M\rightarrow N: 0.1728\times0.20\times0.20=0.006912
$$

$$
\boxed{N_3=0.012544\text{, from }N_2}
$$

### Best route into current state M

$$
N\rightarrow M: 0.0896\times0.30\times0.90=0.024192
$$

$$
M\rightarrow M: 0.1728\times0.80\times0.90=0.124416
$$

$$
\boxed{M_3=0.124416\text{, from }M_2}
$$

| Position 3 state | Best score | Came from |
|---|---:|---|
| `N` | 0.012544 | `N` |
| `M` | 0.124416 | `M` |

## 12. Position 4: observation P

### Best route into current state N

$$
N\rightarrow N: 0.012544\times0.70\times0.80=0.00702464
$$

$$
M\rightarrow N: 0.124416\times0.20\times0.80=0.01990656
$$

$$
\boxed{N_4=0.01990656\text{, from }M_3}
$$

### Best route into current state M

$$
N\rightarrow M: 0.012544\times0.30\times0.10=0.00037632
$$

$$
M\rightarrow M: 0.124416\times0.80\times0.10=0.00995328
$$

$$
\boxed{M_4=0.00995328\text{, from }M_3}
$$

| Position 4 state | Best score | Came from |
|---|---:|---|
| `N` | **0.01990656** | `M` |
| `M` | 0.00995328 | `M` |

The larger final score is the score for `N`, so the most probable path ends in `N`.

## 13. Trace backwards to recover the path

1. Position 4 ends in `N`.
2. `N` at position 4 came from `M` at position 3.
3. `M` at position 3 came from `M` at position 2.
4. `M` at position 2 came from `N` at position 1.

```text
Position:      1  2  3  4
Observation:   P  H  H  P
Best state:    N  M  M  N
```

$$
\boxed{\text{Most probable path}=N\rightarrow M\rightarrow M\rightarrow N}
$$

The two middle hydrophobic observations are predicted to form a membrane region.

## 14. Completed Viterbi table

| Position | Observation | Best N score | Previous for N | Best M score | Previous for M |
|---:|---|---:|---|---:|---|
| 1 | P | 0.6400 | Start | 0.0200 | Start |
| 2 | H | 0.0896 | N | 0.1728 | N |
| 3 | H | 0.012544 | N | 0.124416 | M |
| 4 | P | 0.01990656 | M | 0.00995328 | M |

The scores show the strongest route to each state. The previous-state columns reconstruct the path.

Do not simply select the largest value independently at every position. Begin with the largest score in the final position and trace backwards.

## 15. Evaluating predictions by hand

Suppose a test sequence has:

```text
Position:    1  2  3  4  5  6  7  8
Known:       N  M  M  N  N  M  M  N
Predicted:   N  M  M  N  M  M  N  N
```

| Known state | Predicted N | Predicted M | Total |
|---|---:|---:|---:|
| Known N | 3 | 1 | 4 |
| Known M | 1 | 3 | 4 |

Correct predictions are on the diagonal:

$$
3+3=6
$$

$$
Accuracy=\frac{6}{8}=0.75=75\%
$$

One non-membrane position was incorrectly predicted as membrane, and one membrane position was incorrectly predicted as non-membrane.

## 16. Why this is machine learning

### Training

Labelled protein sequences are used to estimate the initial, emission and transition probabilities.

### Prediction

The trained probabilities are applied to a new observation sequence to infer the most probable hidden-state path.

### Evaluation

Predicted states are compared with known states that were not used to make the prediction.

```text
Labelled training sequences
          ↓
Learn probability tables
          ↓
New protein observations
          ↓
Predict hidden-state path
          ↓
Compare with known test labels
```

## 17. Relationship with previous classifiers

| Model | Main prediction idea |
|---|---|
| Decision tree | Follow feature thresholds |
| Random forest | Combine votes from many trees |
| kNN | Vote among nearby training observations |
| HMM | Find the most probable ordered sequence of hidden states |

The distinctive feature of an HMM is that it models relationships between neighbouring states.

## 18. Short extension: profile HMMs

The HMM in this lecture predicts membrane and non-membrane regions.

A profile HMM asks a different question:

> Does an unknown protein sequence match the pattern of a known protein family or domain?

Profile HMMs use match, insertion and deletion states derived from a multiple-sequence alignment. HMMER and Pfam apply this approach to protein-family and domain prediction.

Detailed profile-HMM searching belongs in a separate protein-bioinformatics practical.

## 19. Real implementation with `hmmlearn`

The hand calculations explain how the HMM works. A computer is used when the sequence is long, when many sequences must be analysed or when the probability tables must be learned from data.

Scikit-learn does not contain an HMM class. The package `hmmlearn` provides HMM models with familiar methods such as:

```python
model.fit(...)
model.predict(...)
```

Install it once in the active Python environment:

```bash
pip install hmmlearn
```

Restart the notebook kernel after installation if the import is not immediately recognised.

### Observation encoding

`CategoricalHMM` requires numerical observations. We will use:

| Biological observation | Number supplied to the model |
|---|---:|
| `H` | 0 |
| `P` | 1 |

The numbers are category labels. They do not mean that `P` is greater than `H`.

In [ ]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import CategoricalHMM

observation_to_number = {
    "H": 0,
    "P": 1
}

state_names = {
    0: "N",
    1: "M"
}

## 20. Use the probability tables already calculated by hand

This first implementation does not train a new model. It places the probability tables from Section 7 into `CategoricalHMM` and allows the package to perform the Viterbi prediction.

The state order is:

```text
State 0 = N
State 1 = M
```

The observation order is:

```text
Observation 0 = H
Observation 1 = P
```

In [ ]:
model = CategoricalHMM(
    n_components=2,
    init_params=""
)

model.n_features = 2

model.startprob_ = np.array([
    0.80,  # Start in N
    0.20   # Start in M
])

model.transmat_ = np.array([
    [0.70, 0.30],  # From N: next N, next M
    [0.20, 0.80]   # From M: next N, next M
])

model.emissionprob_ = np.array([
    [0.20, 0.80],  # In N: emit H, emit P
    [0.90, 0.10]   # In M: emit H, emit P
])

The arrays are the same three tables calculated by hand:

- `startprob_` stores initial probabilities;
- `transmat_` stores transition probabilities;
- `emissionprob_` stores emission probabilities.

`init_params=""` prevents `hmmlearn` from replacing the supplied tables with randomly initialised values.

Now encode the hand-calculation sequence `P H H P` and ask the model to predict its hidden states.

In [ ]:
new_observations = ["P", "H", "H", "P"]

encoded_observations = []

for observation in new_observations:
    number = observation_to_number[observation]
    encoded_observations.append([number])

encoded_observations = np.array(encoded_observations)

predicted_state_numbers = model.predict(encoded_observations)

predicted_states = []

for state_number in predicted_state_numbers:
    predicted_states.append(state_names[state_number])

prediction_table = pd.DataFrame({
    "position": [1, 2, 3, 4],
    "observation": new_observations,
    "predicted_state": predicted_states
})

prediction_table

The software prediction should be:

```text
N  M  M  N
```

This agrees with the state path obtained by hand. Python is not being used to avoid understanding the calculation; it is being used to apply the trained model automatically.

## 21. Train an HMM from observation sequences

In practice, many sequences may be available. The following example trains a two-state categorical HMM using several observation sequences.

The training sequences contain non-hydrophobic regions surrounding hydrophobic regions:

```text
PPHHHHPP
PPPHHHHPP
PPHHHHHPPP
PHHHHPP
...
```

The hidden labels are not supplied to `fit()`. The algorithm attempts to discover two recurring states from the observation patterns. This is unsupervised training.

In [ ]:
training_sequences = [
    "PPHHHHPP",
    "PPPHHHHPP",
    "PPHHHHHPPP",
    "PHHHHPP",
    "PPPHHHPP",
    "PPHHHHPP",
    "PPPPHHHHPP",
    "PPHHHPPP",
    "PPHHHHHPP",
    "PPPHHHHHPPP",
    "PPHHHHPP",
    "PPPHHHPP"
]

encoded_sequences = []
sequence_lengths = []

for sequence in training_sequences:
    encoded_sequence = []

    for observation in sequence:
        number = observation_to_number[observation]
        encoded_sequence.append([number])

    encoded_sequence = np.array(encoded_sequence)
    encoded_sequences.append(encoded_sequence)
    sequence_lengths.append(len(encoded_sequence))

X_train = np.concatenate(encoded_sequences)

Two objects are needed:

- `X_train` contains all encoded observations;
- `sequence_lengths` records where each separate protein sequence ends.

The lengths are essential. Without them, the package would treat the end of one protein and the beginning of the next protein as neighbouring observations in one long sequence.

In [ ]:
learned_model = CategoricalHMM(
    n_components=2,
    n_iter=300,
    random_state=42,
    tol=0.0001
)

learned_model.n_features = 2

learned_model.fit(
    X_train,
    lengths=sequence_lengths
)

### Meaning of the main arguments

- `n_components=2` asks the model to learn two hidden states.
- `n_iter=300` allows up to 300 training iterations.
- `random_state=42` makes the demonstration reproducible.
- `tol=0.0001` allows training to stop when further improvement becomes very small.
- `fit(...)` estimates the probability tables from the training observations.

Inspect the learned emission probabilities first.

In [ ]:
learned_emissions = pd.DataFrame(
    learned_model.emissionprob_,
    columns=["emit_H", "emit_P"],
    index=["state_0", "state_1"]
)

learned_emissions

The state numbers have no automatic biological meaning. `hmmlearn` may call a hydrophobic state `0` in one fitted model and `1` in another.

Interpret the states by examining the learned emissions:

- the state with the larger `emit_H` probability can be interpreted as membrane-like;
- the state with the larger `emit_P` probability can be interpreted as non-membrane-like.

This interpretation must be made before converting numerical state predictions into biological labels.

In [ ]:
learned_transitions = pd.DataFrame(
    learned_model.transmat_,
    columns=["next_state_0", "next_state_1"],
    index=["from_state_0", "from_state_1"]
)

learned_transitions

In [ ]:
test_sequence = "PPHHHPP"

encoded_test = []

for observation in test_sequence:
    number = observation_to_number[observation]
    encoded_test.append([number])

encoded_test = np.array(encoded_test)

learned_state_numbers = learned_model.predict(encoded_test)

learned_prediction_table = pd.DataFrame({
    "position": range(1, len(test_sequence) + 1),
    "observation": list(test_sequence),
    "numerical_state": learned_state_numbers
})

learned_prediction_table

The numerical states must be interpreted using `learned_emissions`. Do not assume that state `0` always means non-membrane or that state `1` always means membrane.

In real work, predictions should also be checked using:

- proteins with experimentally known regions;
- an independent test set;
- biological knowledge about expected membrane-region length;
- additional prediction tools; and
- appropriate evaluation measures.

## 22. When to use each approach

| Situation | Appropriate approach |
|---|---|
| Learning how the probabilities work | Hand calculations |
| Checking a short Viterbi example | Hand calculation followed by `model.predict()` |
| Applying known probability tables to many sequences | `CategoricalHMM` with supplied parameters |
| Discovering states from many unlabelled sequences | `CategoricalHMM.fit()` |
| Protein-family or domain searching | Profile-HMM tools such as HMMER/Pfam |

The hand and software sections have different purposes. Hand calculation develops understanding; software makes the model usable on realistically sized data.

## 23. Summary

- Amino-acid sequences are ordered data.
- `H` and `P` are observable categories.
- `N` and `M` are hidden biological states.
- Initial probabilities describe the likely starting state.
- Emission probabilities connect states to observations.
- Transition probabilities connect neighbouring states.
- The probabilities can be estimated from labelled training sequences.
- Viterbi calculations retain the best route to each state at every position.
- Backtracking recovers the most probable complete path.
- Predictions are evaluated against known test labels.

$$
\boxed{\text{training counts}\rightarrow\text{probabilities}
\rightarrow\text{Viterbi table}\rightarrow\text{state path}
\rightarrow\text{evaluation}}
$$

## 24. Student activities

### Activity 1 — Identify observations and states

State whether each item is an observation or a hidden state: amino acid `L`, hydrophobic category `H`, membrane state `M`, and non-membrane state `N`.

### Activity 2 — Calculate emission probabilities

Membrane-labelled positions contain 72 hydrophobic and 8 non-hydrophobic observations. Calculate the total, $P(H\mid M)$ and $P(P\mid M)$, then confirm that the probabilities sum to 1.

### Activity 3 — Calculate transition probabilities

Among 50 transitions beginning from `N`, 35 end in `N` and 15 end in `M`. Calculate $P(N\mid N)$ and $P(M\mid N)$.

### Activity 4 — Complete a Viterbi calculation

Using the trained model in Section 7, calculate the most probable path for:

```text
P  H  P
```

At every later position, calculate both routes into `N`, both routes into `M`, retain the larger score for each current state, record its previous state, and trace backwards from the largest final score.

### Activity 5 — Evaluate predictions

```text
Known:      N N M M M N M N N M
Predicted:  N M M M N N M N M M
```

Count correct predictions, calculate accuracy, construct a two-by-two comparison table, and count membrane positions incorrectly predicted as non-membrane.